# **02 - SelfCheck-BERTScore**

In [ ]:
# Restart after running this cell.
!pip install -q "transformers==4.36.2" selfcheckgpt sentencepiece "accelerate>=0.26,<1.0" "bitsandbytes>=0.43.0"
!python -m spacy download en_core_web_sm -q

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sentence-transformers 5.7.0 requires tokenizers>=0.19, but you have tokenizers 0.15.2 which is incompatible.
sentence-transformers 5.7.0 requires transformers<6.0.0,>=4.41.0, but you have transformers 4.36.2 which is incompatible.
gradio 6.30.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio-client 2.7.2 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.41.0 requires huggingface-hub<2.0,>=1.32.0, but you have huggingface-hub 0.36.2 which is incompatible.
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to 

In [ ]:
import os, sys
try:
    from google.colab import drive
    drive.mount('/content/drive')
    RESULTS_DIR = "/content/drive/MyDrive/selfcheckgpt_replication_results"
except ImportError:                      # running outside Colab
    RESULTS_DIR = "./selfcheckgpt_results"
os.makedirs(RESULTS_DIR, exist_ok=True)
os.environ["SC_RESULTS_DIR"] = RESULTS_DIR
if RESULTS_DIR not in sys.path: sys.path.insert(0, RESULTS_DIR)
try:
    from sc_common import *            # shared config, dataset, metrics, cache helpers
except ModuleNotFoundError:
    raise FileNotFoundError(f"Upload sc_common.py into {RESULTS_DIR} first.")
describe_device(); ram("after setup")

Mounted at /content/drive


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/evaluation-00000-of-00001-e91191b8f(…):   0%|          | 0.00/2.56M [00:00<?, ?B/s]

Generating evaluation split:   0%|          | 0/238 [00:00<?, ? examples/s]

Using device: cuda
Tesla T4 - 14.6 GB
Results / caches -> /content/drive/MyDrive/selfcheckgpt_replication_results
[RAM after setup] this process 0.8 GB | system used 1.7 / 12.7 GB


In [ ]:
nlp = get_nlp()
from bert_score import BERTScorer
from selfcheckgpt.utils import expand_list1, expand_list2
bs_scorer = BERTScorer(lang="en", rescale_with_baseline=True, device=str(device))   

def bertscore_matrix(sentences, samples):
    # [n_sentences x n_samples]: best-match BERTScore-F1 of each sentence against each sample 
    M = np.zeros((len(sentences), len(samples)))
    for s_i, sample in enumerate(samples):
        sent_sample = [x.text.strip() for x in nlp(sample).sents if len(x) > 3]
        if not sent_sample: continue
        refs  = expand_list1(sentences, len(sent_sample))
        cands = expand_list2(sent_sample, len(sentences))
        _, _, F1 = bs_scorer.score(cands, refs)
        M[:, s_i] = F1.reshape(len(sentences), len(sent_sample)).max(dim=1).values.numpy()
    return M

The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.


0it [00:00, ?it/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [3]:
if VERIFY_AGAINST_PACKAGE:
    from selfcheckgpt.modeling_selfcheck import SelfCheckBERTScore
    r = rows[IDS[0]]; sents, samp = r["gpt3_sentences"][:3], r["gpt3_text_samples"][:3]
    ref = SelfCheckBERTScore(rescale_with_baseline=True).predict(sentences=sents, sampled_passages=samp)
    check_close("BERTScore re-implementation vs package", 1.0 - bertscore_matrix(sents, samp).mean(axis=1), ref, 1e-4)

t0 = time.time()
MATS["bertscore"]["samples"] = run_cached("bertscore_samples", lambda i: bertscore_matrix(rows[i]["gpt3_sentences"], get_samples(i)), IDS, "BERTScore", save_every=10)
print(f"done in {(time.time() - t0) / 60:.1f} min")
register_main("SelfCheck-BERTScore", scores_from_matrices("bertscore", MATS["bertscore"]["samples"]))

if RUN_WIKIBIO_EVIDENCE:
    MATS["bertscore"]["wiki"] = run_cached("bertscore_wiki", lambda i: bertscore_matrix(rows[i]["gpt3_sentences"], get_samples(i, "wiki")), IDS, "BERTScore (WikiBio)", save_every=20)
    RES_WIKI["WikiBio+BERTScore"] = evaluate(scores_from_matrices("bertscore", MATS["bertscore"]["wiki"]))
ram("after BERTScore")

SelfCheck-BERTScore initialized
[OK] BERTScore re-implementation vs package: max |diff| = 0.00e+00 (tol 0.0001)
BERTScore: 0 cached, 238 to compute


BERTScore:   0%|          | 0/238 [00:00<?, ?it/s]

done in 10.7 min
SelfCheck-BERTScore {'NonFact': np.float64(81.2), 'NonFact*': np.float64(45.27), 'Factual': np.float64(43.47), 'Pearson': np.float64(57.34), 'Spearman': np.float64(54.41), 'n_passages': 238}
BERTScore (WikiBio): 0 cached, 238 to compute


BERTScore (WikiBio):   0%|          | 0/238 [00:00<?, ?it/s]

[RAM after BERTScore] this process 2.2 GB | system used 2.9 / 12.7 GB
